In [1]:
import os
import torch
import numpy as np
from astropy.table import Table as aT
from astropy.table import join as aTjoin

In [2]:
from sedflow import flows as F
from sedflow import galaxy as G

/home/chhahn/.conda/envs/gqp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import corner as DFM
# --- plotting ---
import matplotlib as mpl
import matplotlib.pyplot as plt
#mpl.rcParams['text.usetex'] = True
#mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['axes.xmargin'] = 1
mpl.rcParams['xtick.labelsize'] = 'x-large'
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.major.width'] = 1.5
mpl.rcParams['ytick.labelsize'] = 'x-large'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.major.width'] = 1.5
mpl.rcParams['legend.frameon'] = False

In [4]:
lowz = aT.read('/tigress/chhahn/sedflow/alexamon_lowz/LOWZ_z_photometry.csv')

In [5]:
lowz

DESI_TARGET,TARGETID,Z,FLUX_W1,FLUX_W2,FLUX_W3,FLUX_W4,FLUX_IVAR_W1,FLUX_IVAR_W2,FLUX_IVAR_W3,FLUX_IVAR_W4,MAG_R,MAG_R_ERR,MAG_G,MAG_G_ERR,MAG_Z,MAG_Z_ERR
int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
4611686018427387904,2267128157700097,0.03110001061038468,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.550903514829866,0.014660058565197366,20.063104105293387,0.011503651684076388,19.44373204002007,0.043273442333195467
4611686018427387904,2267132943400961,0.0547667397092142,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.628329021925644,0.01746791678788527,20.033670613737243,0.011267399024248807,19.6607372557575,0.04110368080687674
4611686018427387904,2267168028753921,0.022987389705772042,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.720025899238184,0.0252102831926281,19.969890505911998,0.015365632073806363,19.53861311237496,0.04001991463090452
4611686018427387904,2271525801558028,0.22219074965843683,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.913343126033777,0.030840058687517487,21.394349855798833,0.024310194171184737,20.934833167149435,0.06404382219929651
4611686018427387904,2271541295316994,0.1584443469630328,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.622527979640182,0.02445706212336672,21.1585007389924,0.01967434534065528,20.374051744594222,0.054687609810500845
4611686018427387904,2271545963577344,0.0996845241050366,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.714412719184455,0.029615302986959854,20.91516721251483,0.0182917373309271,20.63803011096995,0.04969164049813552
4611686018427387904,2275924212973581,0.37812517146448693,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.990302009527042,0.01935658407940227,21.543847378809406,0.022170120549488574,21.038697572411984,0.04150362029875703
4611686018427387904,2275928403083273,0.039982058551552124,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.815747727932518,0.01047237260038872,20.32174982674902,0.010279431087468474,19.64742090964686,0.01932421234612303
4611686018427387904,2275959113777162,0.12470979669494867,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.221117510046863,0.006452163538102803,19.59434010315296,0.004848958036145144,19.05144617571652,0.013975175346162477


In [6]:
flux_g = 10**((22.5 - lowz['MAG_G']) * 0.4) # nanomaggies
flux_r = 10**((22.5 - lowz['MAG_R']) * 0.4)
flux_z = 10**((22.5 - lowz['MAG_Z']) * 0.4)
fluxes = np.array([flux_g, flux_r, flux_z, lowz['FLUX_W1'], lowz['FLUX_W2']]).T

# 2. compile other measurements
sig_g  = np.abs(flux_g) * np.abs(-0.4 * np.log(10) * lowz['MAG_G_ERR'])
sig_r  = np.abs(flux_r) * np.abs(-0.4 * np.log(10) * lowz['MAG_R_ERR'])
sig_z  = np.abs(flux_z) * np.abs(-0.4 * np.log(10) * lowz['MAG_Z_ERR'])
sig_w1 = lowz['FLUX_IVAR_W1']**-0.5
sig_w2 = lowz['FLUX_IVAR_W2']**-0.5

sig_fluxes = np.array([sig_g, sig_r, sig_z, sig_w1, sig_w2]).T

redshift = lowz['Z']

/tmp/ipykernel_1529645/489755103.py:10: RuntimeWarning: divide by zero encountered in power
  sig_w1 = lowz['FLUX_IVAR_W1']**-0.5
/tmp/ipykernel_1529645/489755103.py:11: RuntimeWarning: divide by zero encountered in power
  sig_w2 = lowz['FLUX_IVAR_W2']**-0.5


## load `DESIflow`

In [7]:
if torch.cuda.is_available(): device = 'cuda'
else: device = 'cpu'

gsed = G.ModelB(name='modelb.lowz', device=device)
desiflow = F.DESIflow(gsed=gsed, device=device)

/home/chhahn/projects/SEDflow/src/sedflow/flows.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.flow = torch.load(fqphi, map_location=self.device)


In [10]:
n_sample = 10000
posterior_samples = np.empty((len(lowz), n_sample, 15))
for i in range(1,5): #len(bgs)):    
    print(i)
    posterior_samples[i,:,:] = desiflow.run(fluxes[-i], sig_fluxes[-i], redshift[-i], 
                                            Nsample=n_sample, progress_bar=False)

1
2


KeyboardInterrupt: 

In [13]:
%timeit posterior_samples[i,:,:] = desiflow.run(fluxes[-i], sig_fluxes[-i], redshift[-i], Nsample=n_sample, progress_bar=False)

156 ms ± 190 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [20]:
import corner as dfm